# Validation of L1 pipeline performance

We here test the L1 pipeline data products for the P1 and P5 sample. The production of the following data was done using the corresponding input catalogues and the PLATOnium executions:

- platonium.py L1-pipeline 500 1 1 23 --seed 1234 --sample P1
- platonium.py L1-pipeline 500 1 1 23 --seed 1234 --sample P5

This, we test with star ID 500, N-CAM 1.1 and Q23. We here secure that the seeds are fixed in the random number generator, so we can reproduce the results.

In [ ]:
# Directory tree to all files
datadir = '/lhome/nicholas/Nextcloud/tests/validation_L1-pipeline/'

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
from colorama import Back, Fore, Style
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
import statsmodels.api as sm
from statsmodels.graphics.gofplots import qqplot

from platosim.simfile import SimFile
from platosim.plot import plotPhotometry
from platosim.utilities import errorcode, normalize

import warnings
warnings.simplefilter("ignore")

# Constants
ppmh = 144   # For a cadence of 25s
model = 'y ~ x'
ms, aa = 3, 0.1

In [ ]:
plt.rc('font',   size=15)          # controls default text sizes
plt.rc('axes',   titlesize=15)     # fontsize of the axes title
plt.rc('axes',   labelsize=15)     # fontsize of the x and y labels
plt.rc('xtick',  labelsize=15)     # fontsize of the tick labels
plt.rc('ytick',  labelsize=15)     # fontsize of the tick labels
plt.rc('legend', fontsize=14)      # legend fontsize
plt.rc('figure', titlesize=15)     # fontsize of the figure title

In [ ]:
def sortlc(data, flux_err=False):
    
    # Convert to days
    data['time'] = data['time']/86400
    
    # Normalize flux and find medina filter
    data['flux']     = normalize(data['flux'])
    data['flux_med'] = median_filter(data['flux'], ppmh) 
    
    # P1 sample has flux errors
    if flux_err: data['flux_err'] = normalize(data['flux_err'])
    
    return data



def colortheme(theme):
    # Select theme and PI color
    if theme == 'r': color = ['tomato', 'red', 'orange']
    if theme == 'g': color = ['limegreen', 'forestgreen', 'yellowgreen']
    if theme == 'b': color = ['royalblue', 'darkcyan', 'dodgerblue']
    if theme == 'm': color = ['deeppink', 'darkviolet', 'm']
    if theme == 'y': color = ['khaki', 'gold', 'orange']
    return color



def plot_modelfit(data, lsFit, model, lsModel='OLS', CI=[0.05], alpha=0.1, theme='b', 
                  x='x', y='y', xlab='x', ylab='y', yerr=False):

    # Select theme and PI color
    color = colortheme(theme)
        
    # Set parameter strings 
    reg, pre = 'x', 'y'
    
    # Select model string using R terminology (Linear model is default)
    string = (r'Adj. R$^{2} \approx$ ' + str(round(lsFit.rsquared_adj,2)) + '\n' +
              r'AIC     $\approx$ '    + str(round(lsFit.aic,1)) + '\n' +
              r'BIC     $\approx$ '    + str(round(lsFit.bic,1)) + '\n' +
              r'Cond.  $\approx$ '     + str(round(lsFit.condition_number, 1)) + '\n')
    
    # Fetch fit coefficients
    theta_val, theta_err = [], []
    for i in range(len(lsFit.params)):
        theta_val, theta_err = round(lsFit.params[i],2), round(lsFit.bse[i],2)
        string += fr'$\theta_{i} \approx {theta_val} \pm {theta_err}$' + '\n'
    
    # Select title and model
    title  = r'Fit model: $y = \epsilon + \theta_0$'
    if model == 'y ~ x': title += fr' + $\theta_1 {x}$'   
    if model == 'y ~ x + z': title += fr' + $\theta_1$ {x} + \theta_2 z$'
    if model == 'y ~ x + I(x**2)': title += fr' + $\theta_1 {x} + \theta_2 {x}^2$'
    if model == 'y ~ x + I(x**2)': title += fr' + $\theta_1 {x} + \theta_2 {x}^2$'
    if model == 'y ~ x + I(np.sin(x))': title += fr' + $\theta_1 {x} + \theta_2 \sin({x})$'            

    # Predict response variable
    n = 100
    xpredict = np.linspace(data[reg].min(), data[reg].max(), n)
    xpredict = np.linspace(data[reg].min(), data[reg].max(), n)
    
    # Select predictions
    ypredict    = lsFit.predict(exog=dict(x=xpredict))
    predictions = lsFit.get_prediction()
    
    # PLOTTING
    fig, ax = plt.subplots(figsize=(13,8))
    
    # plot data
    if yerr:
        ax.errorbar(data[reg], data[pre], yerr=data[yerr], marker='.', color='grey', ls='', alpha=0.1, zorder=0)
        ax.plot(data[reg], data[pre], 'ko', ms=2, alpha=alpha, label="Data", zorder=1)
    else:
        ax.plot(data[reg], data[pre], 'ko', ms=2, alpha=alpha, label="Data", zorder=1)
    
    # Allow more CI intervals
    for i in range(len(CI)):
        df_predictions = predictions.summary_frame(alpha=CI[i])
        df_predictions.index = data.x.values

        # Plot CI
        if len(data) < 1e3:
            ax.fill_between(df_predictions.index, 
                            df_predictions.mean_ci_lower, 
                            df_predictions.mean_ci_upper, 
                            alpha=0.2, color=CI_color[i+1], zorder=2)
        ax.plot(data[reg], df_predictions.mean_ci_lower, '-', c=color[i+1], lw=2, zorder=2, label=str((1-CI[i])*100)+'% CI')
        ax.plot(data[reg], df_predictions.mean_ci_upper, '-', c=color[i+1], lw=2, zorder=2)
    
    # Plot PI -> only relevant for OLS
    if len(data) < 1e3:
        ax.fill_between(df_predictions.index, 
                        df_predictions.obs_ci_lower, 
                        df_predictions.obs_ci_upper, 
                        alpha=0.2, color=PI_color, zorder=3)
    ax.plot(data[reg], df_predictions.obs_ci_lower, '-', c=color[-1], lw=2, zorder=2, label=str((1-CI[0])*100)+'% PI')
    ax.plot(data[reg], df_predictions.obs_ci_upper, '-', c=color[-1], lw=2, zorder=2)
    
    # Plot best fit and data
    ax.plot(df_predictions['mean'], color=color[0], label='Fit', zorder=3)
        
    # Settings
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.set_title(title,fontsize=15)
    ax.legend(fontsize=13, title='',fancybox=True, framealpha=0.8, loc='upper right')
    ax.set_xlim(data[reg].iloc[0], data[reg].iloc[-1])
    
    # Add fit box
    RMS = round(np.sqrt(np.mean( (data[pre]-df_predictions['mean'])**2 )))
    string += f'RMS = {RMS} ppm'
    props = dict(boxstyle='round', facecolor=color[0], alpha=0.3)
    ax.text(1.02, 0.98, string, transform=ax.transAxes, fontsize=14, 
            verticalalignment='top', bbox=props)            
    plt.show()
    
    
    
    
    
def plot_residuals(data, lsFit, theme='b', reg='x', alpha=0.1, lsModel='OLS'):
    
    # Choose correct residuals
    if lsModel == 'OLS':  resid = lsFit.resid
    else: resid = lsFit.resid_pearson
    
    color = colortheme(theme)
    fig, ax = plt.subplots(1,2, figsize=(16,5))

    # Plot residuals squared vs. observations
    ax[0].plot(data[reg], resid**2, 'ko', ms=2, alpha=alpha, zorder=1)
    ax[0].grid(True,   color='grey',   ls='-',  lw=0.5, zorder=2)
    ax[0].axhline(y=0, color=color[0], ls='--', lw=2.0, zorder=3)
    ax[0].set_xlabel(reg)
    ax[0].set_ylabel(r"Residuals squared, $\varepsilon^2$")
    ax[0].set_xlim(data[reg].iloc[0], data[reg].iloc[-1])
    
    # Plot residuals vs. plot
    ax[1].plot(lsFit.fittedvalues, resid, 'ko', ms=2, alpha=alpha, zorder=1)
    ax[1].grid(True,   color='grey',    ls='-',  lw=0.5, zorder=2)
    ax[1].axhline(y=0, color=color[0],  ls='--', lw=2.0, zorder=3)
    ax[1].set_xlabel("Predicted reponse")
    ax[1].set_ylabel(r"Residuals, $\epsilon$")
    ax[1].set_xlim(np.min(lsFit.fittedvalues), np.max(lsFit.fittedvalues))
    plt.show()
    
    
    
    
def plot_standardized_residuals(data, lsFit, K, reg='x', lsModel='OLS'):
    
    # Choose correct residuals
    if lsModel == 'OLS':  resid = lsFit.resid
    else: resid = lsFit.resid_pearson
    
    N = len(data[reg])
    s2 = np.sum(resid**2) / (N-K)
    standardizedResiduals = resid / np.sqrt(s2)
    
    # Plot standardized residuals (QQ-plot)
    fig, ax = plt.subplots(figsize=(10,6))
    qqplot(standardizedResiduals, line='45', ax=ax)
    plt.show()
    

# On-ground light curves (P1 sample)

We start by looking at how well PlatoSim performes purely from it's inbuilt on-board photometry module. This is simply to get a feel for the photometric precision we should expect for both the on-ground and on-board extraction from the L1 pipeline. We will use PIC 13820675 (ID 500 in catalogue), a $P=9.72$ magnitude star, throughout this test of the on-ground pipeline validation. As seen below, the photometric precision is at least below $5000$ ppm.

## Test 1 - Disabled all random noise sources + variable sources

We first test if the L1 pipeline has done a proper job when not including any random noise sources or variable signals. We have specifically disabled the following parameters:
- Varsource
- Read noise
- Shot noise
- CTI
- Cosmics
- Jitter
- TED
- KDA

In the following we model the data with Ordinary Least Squares (OLS) and Weighted Least Squares (WLS). Both methods assumes that the errors on our model fit is Guassian distributed. We use a linear model since we already know that the Transmission Efficiency (TE) is decreasing linearly in time.

In [ ]:
# Load Platosim 
file = datadir + 'test1/P1/pic13820675_Ncam1.1_Q23.hdf5' 
f = SimFile(file)
sim = f.getPhotometry(1)
sim = pd.DataFrame({'time':sim[0], 'flux':sim[2]})
sim = sortlc(sim)

# Load data
lc = pd.read_feather(datadir + 'test1/P1/000000500/000000500_Ncam1.1_Q24.ftr')
lc = sortlc(lc)
lc.head()

In [ ]:
# Plot the input variable source
fig, ax = plt.subplots(2, 1, figsize=(14, 10))
ms, aa = 3, 0.1

ax[0].plot(sim['time'], sim['flux'], 'k.', ms=ms, alpha=aa, label='Q23')
ax[0].plot(sim['time'], sim['flux_med'], 'g-', lw=1, label='1h median')
ax[0].set_xlim(sim['time'].iloc[0], sim['time'].iloc[-1])
ax[0].legend(loc='upper right')
ax[0].set_title('Test 1: PlatoSim')

ax[1].plot(lc['time'], lc['flux'], 'k.', ms=ms, alpha=aa, label='Q23')
ax[1].plot(lc['time'], lc['flux_med'], 'b-', lw=1, label='1h median')
ax[1].set_xlim(lc['time'].iloc[0], lc['time'].iloc[-1])
ax[1].legend(loc='upper right')
ax[1].set_title('Test 1: L1 pipeline')

fig.text(0.5, 0.06, 'Time [days]', ha='center')
fig.text(0.04, 0.5, 'Relative flux [ppm]', va='center', rotation='vertical')
plt.show()

In [ ]:
# OLS model
sim = sim.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
sim['x'] = sim['x'].subtract(sim['x'].min())
lsFit = sm.OLS.from_formula(formula=model, data=sim).fit()
lsFit.summary(alpha=0.05)

# Plot regression model and residuals
plot_modelfit(sim, lsFit, model, theme='g', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(sim, lsFit, reg='x', theme='g')
plot_standardized_residuals(sim, lsFit, K=3, reg='x')

In [ ]:
# WLS modelling
lc = lc.rename(columns={'time':'x', 'flux':'y', 'flux_err':'y_err', 'flux_med':'y_med'})
lc['x'] = lc['x'].subtract(lc['x'].min())
lsFit = sm.WLS.from_formula(formula=model, data=lc, weights=1/lc['y_err']).fit()
lsFit.summary(alpha=0.05)

# Plot regression model and residuals
plot_modelfit(lc, lsFit, model, lsModel='WLS', theme='b', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(lc, lsFit, reg='x', theme='b', lsModel='WLS')
plot_standardized_residuals(lc, lsFit, K=3, reg='x', lsModel='WLS')

## Test 2: Read-noise + Shot-noise + CTI

We see that the RMS level of the photometric precision increases significantly.

In [ ]:
# Load Platosim 
file = datadir + 'test2/P1/pic13820675_Ncam1.1_Q24.hdf5' 
f = SimFile(file)
sim = f.getPhotometry(1)
sim = pd.DataFrame({'time':sim[0], 'flux':sim[2]})
sim = sortlc(sim)

# Load light curves
lc1 = pd.read_feather(datadir + 'test2/P1/000000500/000000500_Ncam1.1_Q24.ftr')
lc1 = sortlc(lc1)

In [ ]:
# Plot the input variable source
fig, ax = plt.subplots(2, 1, figsize=(14, 10))
ms, aa = 3, 0.1

ax[0].plot(sim['time'], sim['flux'], '.', ms=ms, alpha=aa, label='Data Q24', c='k')
ax[0].plot(sim['time'], sim['flux_med'], 'g-', lw=1, label='1h median')
ax[0].set_xlim(sim['time'].iloc[0], sim['time'].iloc[-1])
ax[0].legend(loc='upper right')
ax[0].set_title('Test 1: PlatoSim')

ax[1].plot(lc1['time'], lc1['flux'], '.', ms=ms, alpha=aa, label='Data Q24', c='k')
ax[1].plot(lc1['time'], lc1['flux_med'], '-', c='tomato', lw=1, label='1h median')
ax[1].set_xlim(lc1['time'].iloc[0], lc1['time'].iloc[-1])
ax[1].legend(loc='upper right')
ax[1].set_title('Test 1: L1 pipeline')

fig.text(0.5, 0.04, 'Time [days]', ha='center')
fig.text(0.04, 0.5, 'Relative flux [ppm]', va='center', rotation='vertical')
plt.show()

In [ ]:
# OLS model
sim = sim.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
sim['x'] = sim['x'].subtract(sim['x'].min())
lsFit = sm.OLS.from_formula(formula=model, data=sim).fit()
lsFit.summary(alpha=0.05)

# Plot regression model and residuals
plot_modelfit(sim, lsFit, model, theme='g', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(sim, lsFit, reg='x', theme='g')
plot_standardized_residuals(sim, lsFit, K=3, reg='x')

In [ ]:
# Rename parameters
lc = lc1.rename(columns={'time':'x', 'flux':'y', 'flux_err':'y_err', 'flux_med':'y_med'})
lc['x'] = lc['x'].subtract(lc['x'].min())

# Plot regression model and residuals
lsFit = sm.WLS.from_formula(formula=model, data=lc, weights=1/lc['y_err']).fit()
plot_modelfit(lc, lsFit, model, lsModel='WLS', theme='r', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(lc, lsFit, reg='x', theme='r', lsModel='WLS')
plot_standardized_residuals(sim, lsFit, K=3, reg='x', lsModel='WLS')

## Test 3: Read-noise + Shot-noise + CTI  + Cosmics + Jitter + TED + Variability

In [ ]:
# Load data
var = pd.read_csv(datadir + 'varsource.txt', names=['time', 'dmag'], sep=' ', header=None)
var['time'] = var['time'] / 86400 #- var['time'].iloc[0]/86400
var['flux'] = (10**(-var['dmag']/2.5) - 1) * 1e6 

# Load Platosim 
file = datadir + 'test3/P1/pic13820675_Ncam1.1_Q24.hdf5' 
f = SimFile(file)
sim = f.getPhotometry(1)
sim = pd.DataFrame({'time':sim[0], 'flux':sim[2]})
sim = sortlc(sim)

# Load light curves
lc1 = pd.read_feather(datadir + 'test3/P1/000000500/000000500_Ncam1.1_Q24.ftr')
lc1 = sortlc(lc1)
lc1

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(14, 20))

# Crop for Q24 only
time, flux = var['time'].iloc[len(lc1['time']):], var['flux'].iloc[len(lc1['time']):]

# Plot input variable source
ax[0].plot(time, flux, 'm-', lw=0.5)
ax[0].set_title('Noise-less light curve')
ax[0].set_xlim(time.iloc[0], time.iloc[-1])
ax[0].set_title('Test 3: Noise-less light curve')

# Plot PlatoSim light curve
ax[1].plot(sim['time'], sim['flux'], 'k.', ms=1, alpha=0.6)
ax[1].plot(sim['time'], sim['flux_med'], 'g-', lw=0.5, label='1h median')
ax[1].set_xlim(sim['time'].iloc[0], sim['time'].iloc[-1])
ax[1].set_title('Test 3: PlatoSim')

# Plot L1 pipeline ligth curve
ax[2].plot(lc1['time'], lc1['flux'], 'k.', ms=1, alpha=0.6)
ax[2].plot(lc1['time'], lc1['flux_med'], 'b-', lw=0.5)
ax[2].plot(time, flux, 'm-', lw=0.5)
ax[2].set_ylim(-2e4, 2e4)
ax[2].set_xlim(lc1['time'].iloc[0], lc1['time'].iloc[-1])
ax[2].set_title('Test 3: L1 pipeline')

# Plot L1 pipeline ligth curve
ax[3].plot(lc1['time'], lc1['flux_med'], 'b-', lw=0.5)
ax[3].plot(time, flux, 'm-', lw=0.5)
ax[3].set_xlim(lc1['time'].iloc[0], lc1['time'].iloc[-1])
ax[3].set_title('Test 3: L1 pipeline')

# Settings
fig.text(0.5, 0.08, 'Time [days]', ha='center')
fig.text(0.04, 0.5, 'Relative flux [ppm]', va='center', rotation='vertical')
plt.show()

# On-board light curves (P5 sample)

## Test 1: No Random-noise, Systematic-noise, or Variability

In the following we look at the RMS values of the residuals. The last figure shows some strange discontinuities - almost step-sized flux variations.  

In [ ]:
# Load light curves
lc1 = pd.read_feather(datadir + 'test1/P5/000000500/000000500_Ncam1.1_Q23.ftr')
lc2 = pd.read_feather(datadir + 'test1/P5/000000500/000000500_Ncam2.1_Q23.ftr')
lc3 = pd.read_feather(datadir + 'test1/P5/000000500/000000500_Ncam3.1_Q23.ftr')
lc4 = pd.read_feather(datadir + 'test1/P5/000000500/000000500_Ncam4.1_Q23.ftr')
lc1 = sortlc(lc1)
lc2 = sortlc(lc2)
lc3 = sortlc(lc3)
lc4 = sortlc(lc4)
lc1.head()

In [ ]:
# PlatoSim

# L1 pipeline for 4 camera groups
fig, ax = plt.subplots(1,1,figsize=(15, 10))
ms, aa = 3, 0.1
plt.plot(lc1['time'], lc1['flux']+7000, '.', ms=ms, alpha=aa, c='blue')
plt.plot(lc2['time'], lc2['flux']+3000, '.', ms=ms, alpha=aa, c='orange')
plt.plot(lc3['time'], lc3['flux'], '.',      ms=ms, alpha=aa, c='green')
plt.plot(lc4['time'], lc4['flux']-3000, '.', ms=ms, alpha=aa, c='red')
plt.plot(lc1['time'][0], lc1['flux'][0]+7000, '.', ms=8, label=r'N-CAM 1.1, $r_{OA}$ = 16.83 deg', c='blue')
plt.plot(lc2['time'][0], lc2['flux'][0]+3000, '.', ms=8, label=r'N-CAM 2.1, $r_{OA}$ =  9.43 deg', c='orange')
plt.plot(lc3['time'][0], lc3['flux'][0], '.',      ms=8, label=r'N-CAM 3.1, $r_{OA}$ =  3.69 deg', c='green')
plt.plot(lc4['time'][0], lc4['flux'][0]-3000, '.', ms=8, label=r'N-CAM 4.1, $r_{OA}$ = 14.35 deg', c='red')
plt.plot(lc1['time'], lc1['flux_med']+7000, 'k-', lw=0.5, label='1h median')
plt.plot(lc2['time'], lc2['flux_med']+3000, 'k-', lw=0.5)
plt.plot(lc3['time'], lc3['flux_med'],      'k-', lw=0.5)
plt.plot(lc4['time'], lc4['flux_med']-3000, 'k-', lw=0.5)
plt.xlim(lc1['time'].iloc[0], lc1['time'].iloc[-1])
plt.title('Test 1 - P5 star Q23')
plt.xlabel('Time [days]')
plt.ylabel('Relative flux [ppm]')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Rename parameters
lc = lc1.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
lc['x'] = lc['x'].subtract(lc['x'].min())

# Plot regression model and residuals
lsFit = sm.OLS.from_formula(formula=model, data=lc).fit()
plot_modelfit(lc, lsFit, model, theme='b', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(lc, lsFit, reg='x', theme='b')
plot_standardized_residuals(lc, lsFit, K=3, reg='x')

In [ ]:
# Rename parameters
lc = lc2.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
lc['x'] = lc['x'].subtract(lc['x'].min())

# Plot regression model and residuals
lsFit = sm.OLS.from_formula(formula=model, data=lc).fit()
plot_modelfit(lc, lsFit, model, theme='y', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(lc, lsFit, reg='x', theme='y')
plot_standardized_residuals(lc, lsFit, K=3, reg='x')

In [ ]:
# Rename parameters
lc = lc3.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
lc['x'] = lc['x'].subtract(lc['x'].min())

# Plot regression model and residuals
lsFit = sm.OLS.from_formula(formula=model, data=lc).fit()
plot_modelfit(lc, lsFit, model, theme='g', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(lc, lsFit, reg='x', theme='g')
plot_standardized_residuals(lc, lsFit, K=3, reg='x')

In [ ]:
# Rename parameters
lc = lc4.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
lc['x'] = lc['x'].subtract(lc['x'].min())

# Plot regression model and residuals
lsFit = sm.OLS.from_formula(formula=model, data=lc).fit()
plot_modelfit(lc, lsFit, model, theme='r', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(lc, lsFit, reg='x', theme='r')
plot_standardized_residuals(lc, lsFit, K=3, reg='x')

## Test 2: Read-noise + Shot-noise + CTI

We see that the RMS level of the photometric precision increases significantly.

In [ ]:
# Load Platosim 
file = datadir + 'test2/P5/pic10779490_Ncam1.1_Q24.hdf5' 
f = SimFile(file)
sim = f.getPhotometry(1)
sim = pd.DataFrame({'time':sim[0], 'flux':sim[2]})
sim = sortlc(sim)

# Load light curves
lc1 = pd.read_feather(datadir + 'test2/P1/000000500/000000500_Ncam1.1_Q24.ftr')
lc1 = sortlc(lc1)

In [ ]:
# Plot the input variable source
fig, ax = plt.subplots(2, 1, figsize=(14, 10))
ms, aa = 3, 0.1

ax[0].plot(sim['time'], sim['flux'], '.', ms=ms, alpha=aa, label='Q24', c='k')
ax[0].plot(sim['time'], sim['flux_med'], 'g-', lw=1, label='1h median')
ax[0].set_xlim(sim['time'].iloc[0], sim['time'].iloc[-1])
ax[0].legend(loc='upper right')
ax[0].set_title('Test 2: PlatoSim')

ax[1].plot(lc1['time'], lc1['flux'], '.', ms=ms, alpha=aa, label='Q24', c='k')
ax[1].plot(lc1['time'], lc1['flux_med'], '-', c='tomato', lw=1, label='1h median')
ax[1].set_xlim(lc1['time'].iloc[0], lc1['time'].iloc[-1])
ax[1].legend(loc='upper right')
ax[1].set_title('Test 2: L1 pipeline')

fig.text(0.5, 0.04, 'Time [days]', ha='center')
fig.text(0.04, 0.5, 'Relative flux [ppm]', va='center', rotation='vertical')
plt.show()

In [ ]:
# OLS model
sim = sim.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
sim['x'] = sim['x'].subtract(sim['x'].min())
lsFit = sm.OLS.from_formula(formula=model, data=sim).fit()
lsFit.summary(alpha=0.05)

# Plot regression model and residuals
plot_modelfit(sim, lsFit, model, theme='g', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(sim, lsFit, reg='x', theme='g')
plot_standardized_residuals(sim, lsFit, K=3, reg='x')

In [ ]:
# Rename parameters
lc = lc1.rename(columns={'time':'x', 'flux':'y', 'flux_med':'y_med'})
lc['x'] = lc['x'].subtract(lc['x'].min())

# Plot regression model and residuals
lsFit = sm.OLS.from_formula(formula=model, data=lc).fit()
plot_modelfit(lc, lsFit, model, theme='r', xlab='Time [days]', ylab='Flux [ppm]')
plot_residuals(lc, lsFit, reg='x', theme='r')
plot_standardized_residuals(lc, lsFit, K=3, reg='x')

## Test 3: Read-noise + Shot-noise + CTI + Cosmics + Jitter + TED + Varsource

In [ ]:
# Load data
var = pd.read_csv(datadir + 'varsource.txt', names=['time', 'dmag'], sep=' ', header=None)
var['time'] = var['time'] / 86400 #- var['time'].iloc[0]/86400
var['flux'] = (10**(-var['dmag']/2.5) - 1) * 1e6 

# Load Platosim 
file = datadir + 'test3/P5/pic10779490_Ncam1.1_Q24.hdf5' 
f = SimFile(file)
sim = f.getPhotometry(1)
sim = pd.DataFrame({'time':sim[0], 'flux':sim[2]})
sim = sortlc(sim)

# Load light curves
lc1 = pd.read_feather(datadir + 'test3/P5/000000500/000000500_Ncam1.1_Q24.ftr')
lc1 = sortlc(lc1)

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(14, 20))

# Crop for Q24 only
time, flux = var['time'].iloc[len(lc1['time']):], var['flux'].iloc[len(lc1['time']):]

# Plot input variable source
ax[0].plot(time, flux, 'm-', lw=0.5)
ax[0].set_title('Noise-less light curve')
ax[0].set_xlim(time.iloc[0], time.iloc[-1])
ax[0].set_title('Test 3: Noise-less light curve')

# Plot PlatoSim light curve
ax[1].plot(sim['time'], sim['flux'], 'k.', ms=1, alpha=0.6)
ax[1].plot(sim['time'], sim['flux_med'], 'g-', lw=0.5, label='1h median')
ax[1].set_xlim(sim['time'].iloc[0], sim['time'].iloc[-1])
ax[1].set_title('Test 3: PlatoSim')

# Plot L1 pipeline ligth curve
ax[2].plot(lc1['time'], lc1['flux'], 'k.', ms=1, alpha=0.6)
ax[2].plot(lc1['time'], lc1['flux_med'], 'b-', lw=0.5)
ax[2].plot(time, flux, 'm-', lw=0.5)
ax[2].set_ylim(-2e4, 2e4)
ax[2].set_xlim(lc1['time'].iloc[0], lc1['time'].iloc[-1])
ax[2].set_title('Test 3: L1 pipeline')

# Plot L1 pipeline ligth curve
ax[3].plot(lc1['time'], lc1['flux_med'], 'b-', lw=0.5)
ax[3].plot(time, flux, 'm-', lw=0.5)
ax[3].set_xlim(lc1['time'].iloc[0], lc1['time'].iloc[-1])
ax[3].set_title('Test 3: L1 pipeline - Zoom-in')

# Settings
fig.text(0.5, 0.08, 'Time [days]', ha='center')
fig.text(0.04, 0.5, 'Relative flux [ppm]', va='center', rotation='vertical')
plt.show()